## tl;dr
Обе public схемы согласованы с sample. Classification содержит точные feature-дубликаты, редкие классы и новые категории test. Regression содержит заметный сдвиг пропусков. Эти факты требуют устойчивой валидации, но не доказывают смысловую или временную утечку.

## Context & Methods
Проверка рабочих CSV организаторов, 21 октября 2026. Без обучения и API. Один ряд train/test считается одним наблюдением задачи.

### Key Assumptions
Hash по всем признакам без id/target описывает точные feature-группы. Это не entity/time ID. Скрытые ответы private недоступны.

### Параметры и окружение
Python 3.12, pandas/numpy из фиксированного requirements. Задайте PUBLIC_TASKS_DIR перед запуском kernel. В этой папке должен лежать audit_public_data.py.

In [1]:
from pathlib import Path
import json, os, runpy
assert os.environ.get("PUBLIC_TASKS_DIR"), "Set PUBLIC_TASKS_DIR to the organizer public_tasks folder"
checks = runpy.run_path("audit_public_data.py")
profile_task = checks["profile_task"]
print("Data source:", Path(os.environ["PUBLIC_TASKS_DIR"]).resolve())


Data source: /home/mucg9/Documents/Codex/2026-10-19/referenced-chatgpt-conversation-this-is-an/outputs/automl-hackathon-kit/public_tasks


## Data
Входы: train.csv, test.csv, data_dictionary.csv и sample_submission.csv двух public задач. Проверка читает только локальные файлы.

In [2]:
profiles = {
 "regression": profile_task("regression/regression/participants", "site_eui"),
 "multiclass": profile_task("multiclass/multiclass/participants", "attack_cat")
}
for name, p in profiles.items():
 print(name, "train/test:", p["shapes"]["train"], p["shapes"]["test"], "features:", p["schema"]["features_train"])
 assert p["schema"]["feature_sets_equal"]
 assert p["schema"]["dictionary_matches_train_columns"]
 assert p["ids"]["sample_matches_test_ids"]
 assert p["target_profile"]["null"] == 0
 assert p["ids"]["train_null"] == p["ids"]["test_null"] == 0
 assert p["ids"]["train_duplicate_rows"] == p["ids"]["test_duplicate_rows"] == 0


regression train/test: [53308, 64] [22449, 63] features: 62
multiclass train/test: [87670, 44] [41166, 43] features: 42


## Results
Дубликаты ниже считаются без id/target. Число duplicated rows не равно числу всех строк в повторных группах.

In [3]:
for name, p in profiles.items():
 print(name, "duplicates:", p["duplicates"])
 print(name, "missing rates train/test:", p["missing"]["train_top_rates"], p["missing"]["test_top_rates"])
 print(name, "unseen:", p["categoricals"]["unseen_test_categories"])
print("Classification counts:", profiles["multiclass"]["target_profile"]["class_counts"])
Path("dataset_profile_rechecked.json").write_text(json.dumps(profiles, ensure_ascii=False, indent=2))


regression duplicates: {'train_exact_duplicate_rows': 0, 'test_exact_duplicate_rows': 0, 'train_feature_duplicates_excluding_id_target': 327, 'test_feature_duplicates_excluding_id': 37, 'repeated_train_feature_hashes': 270, 'repeated_train_feature_hashes_with_conflicting_targets': 243, 'train_rows_in_conflicting_feature_groups': 538, 'train_test_shared_exact_feature_hashes': 0, 'train_rows_with_feature_match_in_test': 0, 'test_rows_with_feature_match_in_train': 0, 'test_rows_matching_ambiguous_train_target': 0}
regression missing rates train/test: {'days_with_fog': 0.617693, 'direction_peak_wind_speed': 0.507203, 'max_wind_speed': 0.493528, 'direction_max_wind_speed': 0.493528, 'energy_star_rating': 0.416861, 'year_built': 0.032397} {'max_wind_speed': 0.658069, 'direction_peak_wind_speed': 0.658069, 'direction_max_wind_speed': 0.658069, 'days_with_fog': 0.57321, 'energy_star_rating': 0.199875, 'year_built': 0.0049}
regression unseen: {}
multiclass duplicates: {'train_exact_duplicate_ro

## Takeaways
Использовать grouped validation как диагностику оптимизма дубликатов, per-class F1 и повторные split для редких классов. Для regression проверить missing indicators и срезы. Не считать пересечение id совпадением объектов. Причина противоречивых labels не установлена. Полная интерпретация, severity и остаточные риски — data_audit.md.

Проверка выполнена последовательно в свежем Python-процессе; Jupyter kernel в этой среде не запускался. В notebook сохранены фактические stdout каждой code cell.